# Exercise 3.2: Conversational Memory

**Module:** 3 — LangChain Fundamentals
**Level:** Intermediate

In the previous notebook, you built chains that process one message at a time. But real conversations require **memory** — the AI needs to remember what you said before.

**What you'll do:**
1. See the problem: chains without memory forget everything
2. Add memory so the chain remembers the conversation
3. Build a travel assistant that maintains context across messages
4. Understand how memory works under the hood

**Prerequisite:** Complete 3.1 (Your First Chain) first.

## 1. Setup

Install the packages and set your API key.

Get your free Groq API key at: https://console.groq.com/keys

In [ ]:
# Install LangChain with Groq integration and community extensions
# langchain-community includes the message history utilities
!pip install langchain langchain-groq langchain-community -q

In [ ]:
import os

# Paste your Groq API key between the quotes
# Get it free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. The Problem: Chains Have No Memory

Let's prove that a regular chain forgets everything between calls. We'll send two messages and see what happens.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create a simple travel assistant chain — no memory
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Be concise."),
    ("human", "{input}")
])

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Build the chain: prompt → model → string output
chain = prompt | model | StrOutputParser()

# First message: tell the AI our travel plans
response1 = chain.invoke({"input": "I'm planning a trip to Tokyo next month."})
print("Message 1:", response1)

In [ ]:
# Second message: ask a follow-up question
# The AI should know we're talking about Tokyo, but...
response2 = chain.invoke({"input": "What airlines fly there from Istanbul?"})
print("Message 2:", response2)

# The AI has NO IDEA what "there" means!
# Each invoke() is a completely independent call — no context from message 1.
print("\n--- Notice: the AI doesn't know 'there' means Tokyo. Each call is independent. ---")

## 3. The Solution: Message History

To fix this, we need two things:

1. **ChatMessageHistory** — a container that stores past messages (human + AI)
2. **RunnableWithMessageHistory** — a wrapper that automatically loads/saves history for each call

The prompt template also needs a `{history}` placeholder where previous messages get injected.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Step 1: Create a prompt that includes conversation history.
# MessagesPlaceholder("history") is where all previous messages get inserted.
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Be concise."),
    MessagesPlaceholder("history"),  # <-- Previous messages go here
    ("human", "{input}")            # <-- Current message
])

# Step 2: Build the chain as before
chain_with_history = prompt_with_history | model | StrOutputParser()

print("Prompt now includes a 'history' placeholder for previous messages.")
print("Next: we need a place to STORE those messages.")

In [ ]:
# Step 3: Create a message store.
# This dictionary maps session IDs to their chat history.
# Different users (or different conversations) get different histories.
message_store = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    """Get or create a chat history for a given session."""
    # If this session doesn't exist yet, create a new empty history
    if session_id not in message_store:
        message_store[session_id] = ChatMessageHistory()
    return message_store[session_id]


# Step 4: Wrap the chain with memory.
# RunnableWithMessageHistory handles loading history before each call
# and saving the new messages after each call — automatically.
chain_with_memory = RunnableWithMessageHistory(
    chain_with_history,                    # The chain to wrap
    get_session_history,                   # Function to get/create history
    input_messages_key="input",            # Which key in the input is the user's message
    history_messages_key="history"          # Which placeholder to inject history into
)

print("Chain is now wrapped with memory!")
print("Each call automatically loads previous messages and saves new ones.")

## 4. Testing Memory — The Same Conversation

Let's repeat the same two messages. This time, the AI should remember "Tokyo" from message 1.

In [ ]:
# Configuration: which session are we using?
# The session_id lets us have multiple independent conversations.
config = {"configurable": {"session_id": "user_123"}}

# First message — same as before
response1 = chain_with_memory.invoke(
    {"input": "I'm planning a trip to Tokyo next month."},
    config=config
)
print("Message 1:", response1)

In [ ]:
# Second message — the follow-up
# Now the AI should understand "there" = Tokyo!
response2 = chain_with_memory.invoke(
    {"input": "What airlines fly there from Istanbul?"},
    config=config  # Same session_id = same conversation
)
print("Message 2:", response2)
print("\n--- The AI remembers we're talking about Tokyo! ---")

In [ ]:
# Third message — keep building on the conversation
response3 = chain_with_memory.invoke(
    {"input": "Which one has the best business class?"},
    config=config
)
print("Message 3:", response3)
print("\n--- The AI remembers Tokyo AND the airlines it mentioned! ---")

## 5. Under the Hood — What's Stored

Let's peek inside the message store to see exactly what the memory contains.

In [ ]:
# Get the history for our session
history = get_session_history("user_123")

# Print each message in the history
print(f"=== Message History for session 'user_123' ({len(history.messages)} messages) ===")
print()
for i, msg in enumerate(history.messages):
    # Each message has a type (human/ai) and content (the text)
    role = "HUMAN" if msg.type == "human" else "AI"
    # Truncate long AI responses for readability
    content = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
    print(f"  [{i+1}] {role}: {content}")
    print()

## 6. Multiple Sessions — Independent Conversations

Each `session_id` gets its own separate history. This is how you'd handle multiple users or multiple conversations.

In [ ]:
# Start a completely different conversation with a new session_id
config_new = {"configurable": {"session_id": "user_456"}}

# This user is talking about Paris, not Tokyo
response = chain_with_memory.invoke(
    {"input": "I want to visit Paris for a week."},
    config=config_new
)
print("User 456:", response)

# Meanwhile, user_123's conversation about Tokyo is still intact
response = chain_with_memory.invoke(
    {"input": "How about hotels in the Shinjuku area?"},
    config=config  # Back to user_123
)
print("\nUser 123:", response)
print("\n--- Two conversations, completely independent! ---")

## 7. Building a Travel Advisor with Memory

Let's put it all together into a more polished travel assistant that remembers your preferences and builds on the conversation.

In [ ]:
# A more sophisticated travel assistant prompt
travel_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert travel advisor for Amadeus. You help customers plan trips.

Guidelines:
- Remember all details the customer shares (dates, preferences, budget)
- Build on previous messages — don't repeat information
- Be specific with recommendations (airline names, hotel names, prices)
- Keep responses concise but helpful — 2-3 sentences max"""),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])

# Build the chain with memory — same pattern as before
travel_chain = travel_prompt | model | StrOutputParser()

# Fresh message store for this assistant
travel_store = {}


def get_travel_history(session_id: str) -> ChatMessageHistory:
    if session_id not in travel_store:
        travel_store[session_id] = ChatMessageHistory()
    return travel_store[session_id]


travel_advisor = RunnableWithMessageHistory(
    travel_chain,
    get_travel_history,
    input_messages_key="input",
    history_messages_key="history"
)

print("Travel advisor with memory is ready!")

In [ ]:
# Simulate a multi-turn conversation with the travel advisor
session = {"configurable": {"session_id": "trip_planning_001"}}

# Turn 1: Set the destination and dates
messages = [
    "I want to fly from Istanbul to Barcelona, around mid-May, for 5 days.",
    "My budget is around $1500 total including flights and hotel.",
    "I prefer boutique hotels near the Gothic Quarter.",
    "Can you summarize my trip plan so far?"
]

# Send each message and show the response
for i, msg in enumerate(messages, 1):
    print(f"\n{'='*60}")
    print(f"YOU (turn {i}): {msg}")
    print(f"{'='*60}")
    response = travel_advisor.invoke({"input": msg}, config=session)
    print(f"ADVISOR: {response}")

In [ ]:
# Let's verify the memory — the last response should mention ALL previous details:
# Istanbul → Barcelona, mid-May, 5 days, $1500 budget, boutique hotel, Gothic Quarter

# Check how many messages are stored
history = get_travel_history("trip_planning_001")
print(f"Total messages stored: {len(history.messages)}")
print(f"  - Human messages: {sum(1 for m in history.messages if m.type == 'human')}")
print(f"  - AI messages: {sum(1 for m in history.messages if m.type == 'ai')}")

---

## YOUR TURN: Build Your Own Assistant with Memory

Build a **flight rebooking assistant** that:
1. Asks the customer about their cancelled/delayed flight
2. Remembers their preferences (time, airline, class)
3. Suggests alternatives based on the full conversation

Test it with at least 3 turns of conversation.

In [ ]:
# YOUR CODE HERE
# 1. Create a prompt with system message, MessagesPlaceholder("history"), and {input}
# 2. Build the chain: prompt | model | StrOutputParser()
# 3. Create a message store and get_history function
# 4. Wrap with RunnableWithMessageHistory
# 5. Test with 3+ messages that build on each other



## Key Takeaways

- **Without memory**, every `invoke()` is independent — the AI forgets everything
- **ChatMessageHistory** stores the conversation (human + AI messages)
- **RunnableWithMessageHistory** automatically loads and saves history
- **Session IDs** let you maintain multiple independent conversations
- **MessagesPlaceholder** in the prompt is where history gets injected

**Next:** In Module 4, you'll graduate from chains to **graphs** — where your AI can branch, loop, and make decisions.